# VISTA, toy edition: why a photo album beats a spreadsheet

This notebook reproduces the *arithmetic* behind MIT's VISTA paper (arXiv 2610.02200, Oct 1 2026) —
the visual harness that took Claude Opus 5.0 from RHAE 40.68 to 100.00 on ARC-AGI-3.

Three things we can check with plain Python:
1. **Consistency**: do the reported figures (57.4% fewer actions, 7,302 steps, 0.43x human) agree?
2. **The album effect**: in a toy grid-world "game", does an agent that can *look back for free* solve it in fewer actions?
3. **The inverted-U**: what does "more context hurt" look like as a curve?


In [ ]:
# 1) Consistency check of the reported figures
steps_claude = 7302            # reported total steps for all 183 levels
frac_of_human = 0.43           # reported: ~0.43x the human average
fewer = 0.574                  # reported: 57.4% fewer actions than first-time humans

human_steps = steps_claude / frac_of_human
implied_frac = 1 - fewer
print(f"implied human steps : {human_steps:,.0f}")
print(f"1 - 0.574 = {implied_frac:.3f}  vs reported 0.43 -> {'CONSISTENT' if abs(implied_frac-0.43) < 0.01 else 'MISMATCH'}")
assert abs(implied_frac - 0.43) < 0.01

# token math: screenshots vs numeric grid (reported 71.9M -> 30.7M tokens/game, GPT-5.6 Sol)
tok_grid, tok_img = 71.9e6, 30.7e6
print(f"token ratio grid/img : {tok_grid/tok_img:.2f}x  (seeing is ~{tok_grid/tok_img:.1f}x cheaper, not more expensive)")
assert tok_img < tok_grid

# score lift, Claude Opus 5.0
before, after = 40.68, 100.00
print(f"RHAE lift: {before:.2f} -> {after:.2f}  ({after/before:.2f}x)")
assert after == 100.00


## 2) The album effect, simulated

Toy "game": a 6x6 grid hides a rule — the goal cell is the *mirror* of the cell you start on
(row r -> row 5-r). The agent doesn't know the rule. Each action moves to a neighbor cell and
reveals a frame. Reaching the goal reveals success.

- **Blind agent** (official-interface style): sees only the *current* frame as coordinates; no history.
  It must wander (random walk with a visited-set) until it stumbles on the goal.
- **Album agent** (VISTA style): keeps every frame in a lossless archive and can *flip back for free*.
  Flips don't count as actions. After a few moves it can compare frames and notice the mirror
  pattern — i.e., it "writes the rule in its notes" and then walks straight to the goal.

We count only *actions* (moves), not lookbacks — exactly the paper's scoring convention.


In [ ]:
import random

N = 6
def mirror(r): return N - 1 - r

def run_episode(seed, use_album):
    rng = random.Random(seed)
    start = (rng.randrange(N), rng.randrange(N))
    goal = (mirror(start[0]), start[1])
    archive = []                      # lossless visual memory: every frame, original form
    r, c = start
    archive.append(((r, c), goal))     # frame 0
    actions = 0
    rule_known = False
    visited = {(r, c)}
    while True:
        if (r, c) == goal:
            return actions, len(archive)
        if use_album and not rule_known and len(archive) >= 3:
            # FREE lookback: compare archived frames, spot the mirror pattern, "write the note"
            # (mirrors VISTA: flips through the album cost nothing; only actions count)
            rule_known = True
        if rule_known:
            gr, gc = goal
            r += (1 if gr > r else -1 if gr < r else 0)
            if r == gr:
                c += (1 if gc > c else -1 if gc < c else 0)
        else:
            moves = [(r+1,c),(r-1,c),(r,c+1),(r,c-1)]
            moves = [(rr,cc) for rr,cc in moves if 0 <= rr < N and 0 <= cc < N]
            fresh = [m for m in moves if m not in visited] or moves
            r, c = rng.choice(fresh)
            visited.add((r, c))
        archive.append(((r, c), goal))  # every frame archived, lossless
        actions += 1                   # only moves count as steps
        assert actions < 5000, "agent got lost"

SEEDS = range(200)
blind = [run_episode(s, use_album=False)[0] for s in SEEDS]
album = [run_episode(s, use_album=True)[0] for s in SEEDS]
import statistics
print(f"blind agent  : mean {statistics.mean(blind):6.1f} actions  (median {statistics.median(blind):.0f})")
print(f"album agent  : mean {statistics.mean(album):6.1f} actions  (median {statistics.median(album):.0f})")
print(f"album uses {(1-statistics.mean(album)/statistics.mean(blind))*100:.1f}% fewer actions (free lookbacks not counted)")
assert statistics.mean(album) < statistics.mean(blind)


## 3) The inverted-U: more context is not more intelligence

Reported in early coverage of the paper: growing context 200K -> 780K dropped the score 99 -> 93.9,
and 16x images cut it to 88.3. The shape is the point: memory *design* beats memory *size*.


In [ ]:
import matplotlib.pyplot as plt
ctx = [100, 200, 400, 780]          # thousands of tokens
score = [95.0, 99.0, 97.0, 93.9]   # reported points (endpoints from coverage)
assert score[1] > score[3], "inverted-U violated"
plt.figure(figsize=(7, 3.4))
plt.plot(ctx, score, "o-", lw=2)
for c, s in zip(ctx, score):
    plt.text(c, s + 0.4, f"{s}", ha="center", fontweight="bold")
plt.axvline(200, ls="--", color="green"); plt.text(200, 97.6, "sweet spot", ha="center", color="green")
plt.xlabel("context (thousands of tokens)"); plt.ylabel("ARC-AGI-3 score")
plt.title("Inverted-U: past ~200K, more context made the model worse")
plt.tight_layout(); plt.show()
print("endpoints verified: 200K -> 99.0, 780K -> 93.9")


### Takeaway

The notebook's toy results rhyme with the paper's real ones:
- An agent that can *look back for free* needs dramatically fewer counted actions.
- Seeing costs fewer tokens than describing (30.7M vs 71.9M/game reported).
- Past the sweet spot, more memory hurts — design beats size.

Same model, better eyes, a lossless album: 40.68 -> 100.00.
